In [28]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
from sklearn.decomposition import PCA
import numpy as np
import harmonypy as hm
GENE_PANEL = ["ATOH1","DLL1","DLL4","GFI1","AREG","HES1","HES5","JAG2","NOTCH1","NOTCH2","NOTCH3",
              "OLFM4","LEF1","APCDD1","WNT6","NEUROG3","NEUROD1","KRT20","NEURL1","LGR5"]

# Clustreing leiden no cc correction

In [33]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC0327','CRC0542','CRC0322']
tratt_cercato=['NT','NT72h']
train=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    if sample_name in train_id and trattamento in tratt_cercato:
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])

In [38]:
from func import *
n_hvg=1500
df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample"), sep=":", on_duplicate="first")

# 2) AnnData + raw
adata_full = make_anndata_from_df(df_clean, set_raw=True)

# 3) cell cycle
score_cell_cycle(adata_full)

# 4) HVG + PCA
select_hvg_cell_ranger(adata_full, n_top_genes=n_hvg, batch_key="sample", subset=True)
scale_and_pca(adata_full, n_comps=42, max_value=10, random_state=42)
adata_base = adata_full

BEST = dict(
    n_pcs=20,          # PC da usare per BBKNN
    nwb=3,             # neighbors_within_batch
    trim=None,         # None oppure un intero (es. 12)
    leiden_res=0.3,    # granularità (resolution) per Leiden
    out_root="./bbknn_NT_WT"
)


# cartella output
def safe(s): return re.sub(r"[^A-Za-z0-9_=.,+-]", "_", str(s))
outdir = os.path.join(
    BEST["out_root"],
    f"npcs={BEST['n_pcs']}__nwb={BEST['nwb']}__trim={BEST['trim']}__res={BEST['leiden_res']}"
)
os.makedirs(outdir, exist_ok=True)
sc.settings.figdir = outdir

# 5)integrazione
adx = adata_base.copy()
bbknn.bbknn(
    adx,
    batch_key="sample",
    neighbors_within_batch=BEST['nwb'],
    trim=BEST["trim"],
    n_pcs=BEST["n_pcs"]
)
sc.tl.umap(adx, min_dist=0.3, random_state=42)  # UMAP usa il grafo BBKNN

# 6)clustering (Leiden)
clu_key = f"leiden_r{BEST['leiden_res']}"
sc.tl.leiden(adx, resolution=BEST["leiden_res"], key_added=clu_key, random_state=42)

# plot rapidi &
sc.pl.umap(adx, color=["sample", clu_key], ncols=2, wspace=0.3, frameon=False,
           show=False, save="_sample+cluster.png")
save_umap_gene_panel(adx, outdir, GENE_PANEL)
if "phase" in adx.obs:
    sc.pl.umap(adx, color=["phase"], frameon=False, show=False, save="_phase.png")

#adx.write(os.path.join(outdir, "adata_bbknn_final.h5ad"), compression="lzf")





saving figure to file bbknn_NT_WT/npcs=20__nwb=3__trim=None__res=0.3/umap_sample+cluster.png
saving figure to file bbknn_NT_WT/npcs=20__nwb=3__trim=None__res=0.3/umap_genes_panel.png
saving figure to file bbknn_NT_WT/npcs=20__nwb=3__trim=None__res=0.3/umap_phase.png


In [35]:

sc.tl.rank_genes_groups(adx, groupby=clu_key, method="wilcoxon", use_raw=adx.raw is not None)
sc.pl.rank_genes_groups(adx, n_genes=25, sharey=False, show=False, save=f"_{clu_key}_top25.png")
data_marker=sc.get.rank_genes_groups_df(adx, None)

#.to_csv(os.path.join(outdir, f"markers_{clu_key}_all.csv"), index=False)


saving figure to file bbknn_NT_WT/npcs=20__nwb=3__trim=None__res=0.3/rank_genes_groups_leiden_r0.3_leiden_r0.3_top25.png


In [37]:
data_obs = pd.DataFrame(adx.obs)


data_obs["umap_1"] = adx.obsm["X_umap"][:, 0]
data_obs["umap_2"] = adx.obsm["X_umap"][:, 1]

data_obs['trattamento']='NT'
data_obs.to_csv(
    path_or_buf="bbknn_NT_WT/npcs=20__nwb=3__trim=None__res=0.3/obs_data_with_umap.csv",
    index=False
)

In [79]:
df_all_sorted = (data_marker
                 .sort_values(["group", "logfoldchanges"], ascending=[True, False], na_position="last")
                )

outdir_mark = os.path.join(outdir, f"markers_{clu_key}_FULL")
os.makedirs(outdir_mark, exist_ok=True)

for grp, df_g in df_all_sorted.groupby("group", sort=False):
    sub = df_g[["names", "logfoldchanges"]].rename(
        columns={"names": "gene", "logfoldchanges": "logfoldchange"}
    )
    sub.to_csv(os.path.join(outdir_mark, f"markers_{clu_key}__{safe(grp)}.csv"), index=False)
df_all_simple = df_all_sorted[["group", "names", "logfoldchanges"]].rename(
    columns={"names": "gene", "logfoldchanges": "logfoldchange"}
)
df_all_simple.to_csv(os.path.join(outdir_mark, f"markers_{clu_key}__ALL.csv"), index=False)

/tmp/ipykernel_3330295/1491903864.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp, df_g in df_all_sorted.groupby("group", sort=False):


# Clustering Leiden with cc correction

In [17]:
from func import *
import scanpy as sc, bbknn, os, re

# --- parametri ---
n_hvg = 1500
BEST = dict(
    n_pcs=20,          
    nwb=3,             
    trim=None,         
    leiden_res=0.2,
    out_root="./bbknn_NT_WT_cc",
    louvain_res=0.3
    
)

# 1) pulizia nomi + AnnData con raw
df_clean, dup = strip_prefix_from_genes(train, meta_cols=("cell_id","sample"), sep=":", on_duplicate="first")
adata_full_ = make_anndata_from_df(df_clean, set_raw=True)

# 2) punteggi/fase ciclo (prima di HVG)
score_cell_cycle(adata_full)

# 3) base corretta per ciclo:

adata = make_base_cc_regressed(adata_full, n_hvg=n_hvg, batch_key="sample", n_pcs= BEST["n_pcs"], rng=42)

# 4) integrazione BBKNN 
bbknn.bbknn(adata, batch_key="sample",
            neighbors_within_batch=BEST["nwb"],
            trim=BEST["trim"],
            n_pcs=BEST["n_pcs"])
sc.tl.umap(adata, min_dist=0.3, random_state=42)

# 5) clustering Leiden
#clu_key = f"leiden_cc_r{BEST['leiden_res']}"
#sc.tl.leiden(adata, resolution=BEST["leiden_res"], key_added=clu_key, random_state=42)
# 6) clustering louvain
clu_key = f"louvain_cc_r{BEST['louvain_res']}"
sc.tl.louvain(adata, resolution=BEST["louvain_res"], key_added=clu_key, random_state=42)


def safe(s): return re.sub(r"[^A-Za-z0-9_=.,+-]", "_", str(s))
outdir = os.path.join(BEST["out_root"], f"npcs={BEST['n_pcs']}__nwb={BEST['nwb']}__trim={BEST['trim']}__res={BEST['leiden_res']}")
os.makedirs(outdir, exist_ok=True)
sc.settings.figdir = outdir

sc.pl.umap(adata, color=["sample", "phase", clu_key], ncols=3, wspace=0.3, frameon=False,
           show=False, save="_sample_phase_cluster.png")

#adata.write(os.path.join(outdir, "adata_bbknn_cc_final.h5ad"), compression="lzf")

#  marker su tutti i geni
# sc.tl.rank_genes_groups(adata, groupby=clu_key, method="wilcoxon", use_raw=True)
# sc.get.rank_genes_groups_df(adata, None).to_csv(os.path.join(outdir, f"markers_{clu_key}_ALL.csv"), index=False)

#adx.write(os.path.join(outdir, "adata_bbknn_final.h5ad"), compression="lzf")



saving figure to file bbknn_NT_WT_cc/npcs=20__nwb=3__trim=None__res=0.2/umap_sample_phase_cluster.png


[<Axes: title={'center': 'sample'}, xlabel='UMAP1', ylabel='UMAP2'>,
 <Axes: title={'center': 'phase'}, xlabel='UMAP1', ylabel='UMAP2'>,
 <Axes: title={'center': 'louvain_cc_r0.3'}, xlabel='UMAP1', ylabel='UMAP2'>]

In [ ]:
ad_no = adx     
ad_cc =adata      

clu_key_no = "leiden_r0.3"       
clu_key_cc = "leiden_cc_r0.2"


cells = ad_no.obs_names.intersection(ad_cc.obs_names)
lab_no = ad_no.obs[clu_key_no].loc[cells].astype(str)
lab_cc= ad_cc.obs[clu_key_cc].loc[cells].astype(str)

df_assign = pd.DataFrame({
    "uid": cells.astype(str),
    "cluster_no_cc": lab_no.values,
    "cluster_cc": lab_cc.values
}).set_index("uid")

cm = pd.crosstab(df_assign["cluster_no_cc"], df_assign["cluster_cc"])
cm.to_csv("confusion_noCC_vs_CC_counts.csv")
display(cm)



In [82]:
import pandas as pd
import scanpy as sc


clu_key_no = "leiden_r0.3"       
clu_key_cc = "leiden_cc_r0.2"


labels_cc_on_no = ad_cc.obs[clu_key_cc].astype(str).reindex(ad_no.obs_names)
ad_no.obs["cluster_cc_on_no"] = labels_cc_on_no.values
ad_no.obs["cluster_cc_on_no"] = ad_no.obs["cluster_cc_on_no"].astype("category")


sc.pl.umap(ad_no,
           color=[clu_key_no, "cluster_cc_on_no"],
           ncols=2, wspace=0.3, frameon=False,
           na_color="lightgrey",  # nel caso manchino celle
           show=False, save="_noCCumap_colored_by_noCC_and_CC.png")


saving figure to file bbknn_NT_WT_cc/npcs=20__nwb=3__trim=None__res=0.2/umap_noCCumap_colored_by_noCC_and_CC.png


[<Axes: title={'center': 'leiden_r0.3'}, xlabel='UMAP1', ylabel='UMAP2'>,
 <Axes: title={'center': 'cluster_cc_on_no'}, xlabel='UMAP1', ylabel='UMAP2'>]

# KRAS cetux

In [32]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC1502','CRC1620','CRC1139']
tratt_cercato=['cetux','CTX72h']
train=pd.DataFrame()
for file in os.listdir(data_path):
    
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    if sample_name in train_id and trattamento in tratt_cercato:
        print(file)
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])

filtered_annotated_saver_ribomito_CRC1502_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC1139_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC1620_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC1502_cetux_1.csv


,ENSG00000000003:TSPAN6,ENSG00000000005:TNMD,ENSG00000000419:DPM1,ENSG00000000457:SCYL3,ENSG00000000460:C1orf112,ENSG00000000938:FGR,ENSG00000000971:CFH,ENSG00000001036:FUCA2,ENSG00000001084:GCLC,ENSG00000001167:NFYA,...,ENSG00000282815:TEX13C,ENSG00000282872:C1orf232,ENSG00000282881:TMEM275,ENSG00000282936:AC004706.3,ENSG00000282988:AL031777.2,ENSG00000283039:KLF18,ENSG00000283071:LBHD2,ENSG00000283093:CENPVL2,sample,cell_id
0,7.738531,1.086670,7.325744,3.740082,4.132281,0.0,0.0,6.679002,5.460308,4.492710,...,0.214806,0.0,0.214806,0.214806,2.018066,0.0,0.214806,0.0,CRC1502,AAACCCAAGGTGCGAT.1
1,8.741125,1.922616,7.294827,4.894737,2.807533,0.0,0.0,6.760984,6.670985,4.055730,...,0.188471,0.0,0.188471,0.188471,2.073739,0.0,0.188471,0.0,CRC1502,AAACCCACAACGGTAG.1
2,8.112822,4.584078,6.596954,3.943709,4.253414,0.0,0.0,6.766966,5.951510,4.368177,...,0.233226,0.0,0.233226,0.233226,2.280993,0.0,0.233226,0.0,CRC1502,AAACCCACAAGGTCAG.1
3,8.193446,2.823299,7.207522,4.156300,5.147339,0.0,0.0,6.971736,5.454709,4.181300,...,0.208943,0.0,0.208943,0.208943,2.041529,0.0,0.208943,0.0,CRC1502,AAACCCACATTCAGGT.1
4,8.845929,3.118555,6.851617,4.886213,4.257699,0.0,0.0,6.451334,5.904958,4.540232,...,0.259490,0.0,0.259490,0.479346,2.518368,0.0,0.259490,0.0,CRC1502,AAACCCAGTATTGGCT.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5278,2.053000,0.017000,0.849000,0.063000,0.073000,0.0,0.0,0.775000,0.408000,0.113000,...,0.001000,0.0,0.001000,0.001000,0.023000,0.0,0.001000,0.0,CRC1502,TTTGTTGCATGACTCA.1
5279,2.777000,0.268000,0.783000,0.084000,0.080000,0.0,0.0,0.552000,0.360000,0.112000,...,0.001000,0.0,0.001000,0.001000,0.022000,0.0,0.001000,0.0,CRC1502,TTTGTTGGTATGAAAC.1
5280,1.470000,0.021000,0.932000,0.074000,0.185000,0.0,0.0,0.594000,0.350000,0.112000,...,0.001000,0.0,0.001000,0.001000,0.022000,0.0,0.001000,0.0,CRC1502,TTTGTTGGTCCCGGTA.1
5281,0.956000,0.007000,0.810000,0.087000,0.049000,0.0,0.0,0.443000,0.361000,0.114000,...,0.001000,0.0,0.001000,0.002000,0.025000,0.0,0.001000,0.0,CRC1502,TTTGTTGGTCCTGTTC.1


In [30]:
from func import *
n_hvg=1500
df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample"), sep=":", on_duplicate="first")

# 2) AnnData + raw
adata_full = make_anndata_from_df(df_clean, set_raw=True)

# 3) cell cycle
score_cell_cycle(adata_full)

# 4) HVG + PCA
select_hvg_cell_ranger(adata_full, n_top_genes=n_hvg, batch_key="sample", subset=True)
scale_and_pca(adata_full, n_comps=42, max_value=10, random_state=42)
adata_base = adata_full

BEST = dict(
    n_pcs=20,          # PC da usare per BBKNN
    nwb=3,             # neighbors_within_batch
    trim=None,         # None oppure un intero (es. 12)
    leiden_res=0.3,    # granularità (resolution) per Leiden
    out_root="./bbknn_NT_KRAS_prova"
)


# cartella output
def safe(s): return re.sub(r"[^A-Za-z0-9_=.,+-]", "_", str(s))
outdir = os.path.join(
    BEST["out_root"],
    f"npcs={BEST['n_pcs']}__nwb={BEST['nwb']}__trim={BEST['trim']}__res={BEST['leiden_res']}"
)
os.makedirs(outdir, exist_ok=True)
sc.settings.figdir = outdir

# 5)integrazione
adx = adata_base.copy()
bbknn.bbknn(
    adx,
    batch_key="sample",
    neighbors_within_batch=BEST['nwb'],
    trim=BEST["trim"],
    n_pcs=BEST["n_pcs"]
)
sc.tl.umap(adx, min_dist=0.3, random_state=42)  # UMAP usa il grafo BBKNN
save_umap_gene_panel(adx, outdir, GENE_PANEL)
# 6)clustering (Leiden)
clu_key = f"leiden_r{BEST['leiden_res']}"
sc.tl.leiden(adx, resolution=BEST["leiden_res"], key_added=clu_key, random_state=42)

# plot rapidi &
sc.pl.umap(adx, color=["sample", clu_key], ncols=2, wspace=0.3, frameon=False,
           show=False, save="_sample+cluster.png")
if "phase" in adx.obs:
    sc.pl.umap(adx, color=["phase"], frameon=False, show=False, save="_phase.png")

/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/sklearn/manifold/_spectral_

In [21]:
sc.tl.rank_genes_groups(adx, groupby=clu_key, method="wilcoxon", use_raw=adx.raw is not None)
sc.pl.rank_genes_groups(adx, n_genes=25, sharey=False, show=False, save=f"_{clu_key}_top25.png")
data_marker=sc.get.rank_genes_groups_df(adx, None)


saving figure to file bbknn_NT_KRAS/npcs=20__nwb=3__trim=None__res=0.3/rank_genes_groups_leiden_r0.3_leiden_r0.3_top25.png


In [22]:
df_all_sorted = (data_marker
                 .sort_values(["group", "logfoldchanges"], ascending=[True, False], na_position="last")
                )

outdir_mark = os.path.join(outdir, f"markers_{clu_key}_FULL")
os.makedirs(outdir_mark, exist_ok=True)

for grp, df_g in df_all_sorted.groupby("group", sort=False):
    sub = df_g[["names", "logfoldchanges"]].rename(
        columns={"names": "gene", "logfoldchanges": "logfoldchange"}
    )
    sub.to_csv(os.path.join(outdir_mark, f"markers_{clu_key}__{safe(grp)}.csv"), index=False)
df_all_simple = df_all_sorted[["group", "names", "logfoldchanges"]].rename(
    columns={"names": "gene", "logfoldchanges": "logfoldchange"}
)
df_all_simple.to_csv(os.path.join(outdir_mark, f"markers_{clu_key}__ALL.csv"), index=False)

/tmp/ipykernel_3380619/1491903864.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp, df_g in df_all_sorted.groupby("group", sort=False):


In [24]:
data_obs=pd.DataFrame(adx.obs)

In [26]:
data_obs.to_csv(path_or_buf='bbknn_NT_KRAS/npcs=20__nwb=3__trim=None__res=0.3/obs_data.csv',index=False)